1. Configuration

In [ ]:
import os, time, csv, json, numpy as np, cv2
import IPython.display as disp
from pynq_dpu import DpuOverlay
OVERLAY_PATH = '/home/xilinx/TESTES_FINAIS/overlays/B512/dpu.bit'
MODELS = {'3-band': '/home/xilinx/TESTES_FINAIS/compiled/encoder_96_3b_w25_B512/encoder_96_3b_w25_B512.xmodel', '4-band R->NIR': '/home/xilinx/TESTES_FINAIS/compiled/encoder_96_4b_w25__B512/encoder_96_4b_w25__B512.xmodel'}
DATA_DIR = '/home/xilinx/TESTES_FINAIS/test_data'
DECISION_THR = 0.5
N_PER_BUCKET = 2
DEVICE = '/dev/video0'
CAP_W, CAP_H = (640, 480)
SIZE = 96
OUT_CSV = 'camera_test_results.csv'
print('OpenCV', cv2.__version__)


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))
overlay = DpuOverlay(OVERLAY_PATH)
print('overlay loaded:', OVERLAY_PATH)

class Model:

    def __init__(self, xmodel):
        overlay.load_model(xmodel)
        self.r = overlay.runner
        it, ot = (self.r.get_input_tensors()[0], self.r.get_output_tensors()[0])
        self.in_scale = 2 ** it.get_attr('fix_point')
        self.out_scale = 2 ** ot.get_attr('fix_point')
        self.in_shape, self.out_shape = (tuple(it.dims), tuple(ot.dims))
        self.size, self.bands = (self.in_shape[1], self.in_shape[3])
        assert int(np.prod(self.out_shape[1:])) <= 2, 'expected a 1-logit classifier'
        self.ib = [np.zeros(self.in_shape, np.int8)]
        self.ob = [np.zeros(self.out_shape, np.int8)]

    def prob(self, hwc_float):
        self.ib[0][0] = np.clip(np.round(hwc_float * self.in_scale), -128, 127).astype(np.int8)
        j = self.r.execute_async(self.ib, self.ob)
        self.r.wait(j)
        return sigmoid(float(self.ob[0].astype(np.float32).flatten()[0]) / self.out_scale)

def stack_bands(r, g, b, bands):
    chans = [r, g, b] + ([r] if bands == 4 else [])
    return np.stack(chans, axis=-1).astype(np.float32)

def direct_input(patch, bands):
    r, g, b = patch['direct']
    return stack_bands(r, g, b, bands) / 255.0

def camera_input(patch, bands):
    rgb = patch['camera'].astype(np.float32)
    return stack_bands(rgb[..., 0], rgb[..., 1], rgb[..., 2], bands)

def read_band16(stem, band):
    return cv2.imread(os.path.join(DATA_DIR, f'train_{band}', f'{band}_{stem}'), cv2.IMREAD_UNCHANGED).astype(np.float32)

def preview8(stem):
    ch = []
    for b in ('red', 'green', 'blue'):
        a = read_band16(stem, b)
        lo, hi = (np.percentile(a, 2), np.percentile(a, 98))
        ch.append(np.clip((a - lo) / max(hi - lo, 1.0) * 255, 0, 255))
    return np.stack(ch, axis=-1).astype(np.uint8)


2. The 20 test patches

In [ ]:
PATCHES = [('patch_274_14_by_14_LC08_L1TP_039034_20160320_20170224_01_T1.TIF', 0, 0.0, 0), ('patch_451_20_by_14_LC08_L1TP_059014_20160620_20170221_01_T1.TIF', 0, 0.0861, 0), ('patch_45_3_by_5_LC08_L1TP_002054_20160520_20170324_01_T1.TIF', 10, 0.1014, 0), ('patch_183_9_by_15_LC08_L1TP_045026_20160720_20170221_01_T1.TIF', 10, 0.1986, 0), ('patch_330_16_by_15_LC08_L1TP_029040_20160720_20170222_01_T1.TIF', 20, 0.2008, 0), ('patch_301_15_by_7_LC08_L1TP_029040_20160720_20170222_01_T1.TIF', 20, 0.2954, 0), ('patch_287_14_by_14_LC08_L1TP_034034_20160520_20170223_01_T1.TIF', 30, 0.3041, 1), ('patch_346_17_by_10_LC08_L1TP_029040_20160720_20170222_01_T1.TIF', 30, 0.3973, 1), ('patch_353_18_by_13_LC08_L1TP_002053_20160520_20170324_01_T1.TIF', 40, 0.4033, 1), ('patch_124_7_by_4_LC08_L1TP_039034_20160320_20170224_01_T1.TIF', 40, 0.4985, 1), ('patch_207_10_by_18_LC08_L1TP_034034_20160520_20170223_01_T1.TIF', 50, 0.5011, 1), ('patch_154_8_by_14_LC08_L1TP_035034_20160120_20170224_01_T1.TIF', 50, 0.5997, 1), ('patch_194_10_by_5_LC08_L1TP_034034_20160520_20170223_01_T1.TIF', 60, 0.6032, 1), ('patch_190_10_by_10_LC08_L1TP_035034_20160120_20170224_01_T1.TIF', 60, 0.6998, 1), ('patch_86_5_by_6_LC08_L1TP_039034_20160320_20170224_01_T1.TIF', 70, 0.7177, 1), ('patch_282_15_by_2_LC08_L1TP_002054_20160520_20170324_01_T1.TIF', 70, 0.7997, 1), ('patch_289_14_by_16_LC08_L1TP_011247_20160620_20170323_01_T1.TIF', 80, 0.8067, 1), ('patch_450_20_by_13_LC08_L1TP_059014_20160620_20170221_01_T1.TIF', 80, 0.8967, 1), ('patch_228_12_by_8_LC08_L1TP_035034_20160120_20170224_01_T1.TIF', 90, 0.9051, 1), ('patch_451_20_by_14_LC08_L1TP_044010_20160220_20170224_01_T1.TIF', 90, 1.0, 1)]
selected = []
for stem, bp, frac, gt in PATCHES:
    try:
        rgb16 = [cv2.resize(read_band16(stem, b), (SIZE, SIZE), interpolation=cv2.INTER_AREA) for b in ('red', 'green', 'blue')]
    except Exception as e:
        print(f'  MISSING on board, skipped: {stem} ({e})')
        continue
    selected.append({'stem': stem, 'bucket': bp // 10, 'frac': frac, 'gt': gt, 'direct': rgb16, 'camera': None})
print(f'{len(selected)}/{len(PATCHES)} patches loaded (clean test split, 2 per 10% bucket)')
for p in selected:
    print(f"  [{p['bucket'] * 10:3d}-{p['bucket'] * 10 + 10:3d}%]  frac={p['frac']:.2f}  GT={('DISCARD' if p['gt'] else 'KEEP')}  {p['stem']}")


3. Direct test (no camera)

In [ ]:
def run_phase(builder, label):
    out = {}
    print(f'{label}:')
    for name, path in MODELS.items():
        m = Model(path)
        probs, decs, correct = ([], [], 0)
        for p in selected:
            pr = m.prob(builder(p, m.bands))
            d = 1 if pr >= DECISION_THR else 0
            probs.append(pr)
            decs.append(d)
            correct += d == p['gt']
        out[name] = {'prob': probs, 'dec': decs, 'acc': correct / len(selected)}
        print(f"  {name:14s} accuracy vs GT = {out[name]['acc']:.3f}")
    return out
DIRECT_JSON = 'camera_test_direct.json'
if os.path.exists(DIRECT_JSON):
    direct_res = json.load(open(DIRECT_JSON))
    print('loaded cached direct results from', DIRECT_JSON)
    for n in MODELS:
        print(f"  {n:14s} accuracy vs GT = {direct_res[n]['acc']:.3f}")
else:
    direct_res = run_phase(direct_input, 'DIRECT (dataset pixels)')
    json.dump(direct_res, open(DIRECT_JSON, 'w'))
    print('saved ->', DIRECT_JSON, '(delete it to recompute)')


4. Manual capture of patches (host pc running `display_patches.py`)

In [ ]:
import ipywidgets as wdg
from IPython.display import display

def open_cap():
    c = cv2.VideoCapture(DEVICE, cv2.CAP_V4L2)
    c.set(cv2.CAP_PROP_FRAME_WIDTH, CAP_W)
    c.set(cv2.CAP_PROP_FRAME_HEIGHT, CAP_H)
    c.set(cv2.CAP_PROP_BUFFERSIZE, 2)
    return c

def capture_rgb(cap):
    ok, bgr = cap.read()
    if not ok:
        raise RuntimeError('capture failed')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def center_crop_square(img):
    h, w = img.shape[:2]
    s = min(h, w)
    y, x = ((h - s) // 2, (w - s) // 2)
    return img[y:y + s, x:x + s]

def _jpg(rgb, scale):
    big = cv2.resize(rgb, (scale, scale), interpolation=cv2.INTER_NEAREST)
    ok, j = cv2.imencode('.jpg', cv2.cvtColor(big, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 85])
    return j.tobytes()
cap = open_cap()
for _ in range(5):
    capture_rgb(cap)
st = {'i': 0}
feed = wdg.Image(format='jpeg', width=360)
lbl = wdg.Label()

def _status():
    i, done = (st['i'], sum((p['camera'] is not None for p in selected)))
    if i < len(selected):
        p = selected[i]
        return f"[{i + 1}/{len(selected)}]  display: {p['stem']}   (bucket {p['bucket'] * 10}%, GT={('DISCARD' if p['gt'] else 'KEEP')})   captured {done}/{len(selected)}"
    return f'captured {done}/{len(selected)} -- done, release the camera in the last cell'

def _live():
    for _ in range(2):
        f = capture_rgb(cap)
    g = f.copy()
    h, w = g.shape[:2]
    s = min(h, w)
    y, x = ((h - s) // 2, (w - s) // 2)
    cv2.rectangle(g, (x, y), (x + s, y + s), (0, 255, 0), 2)
    feed.value = _jpg(g, 360)
    return f

def on_prev(_):
    _live()

def on_cap(_):
    i = st['i']
    if i >= len(selected):
        return
    f = _live()
    selected[i]['camera'] = cv2.resize(center_crop_square(f), (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    feed.value = _jpg(selected[i]['camera'], 192)
    st['i'] += 1
    lbl.value = _status()

def on_skip(_):
    if st['i'] < len(selected):
        st['i'] += 1
    lbl.value = _status()

def on_back(_):
    if st['i'] > 0:
        st['i'] -= 1
    lbl.value = _status()
bp = wdg.Button(description='Preview', button_style='info')
bc = wdg.Button(description='Capture', button_style='success')
bs = wdg.Button(description='Skip')
bb = wdg.Button(description='Back')
bp.on_click(on_prev)
bc.on_click(on_cap)
bs.on_click(on_skip)
bb.on_click(on_back)
lbl.value = _status()
display(wdg.VBox([lbl, wdg.HBox([bp, bc, bs, bb]), feed]))


5. Camera test

In [ ]:
assert any((p['camera'] is not None for p in selected)), 'capture some frames first'
camera_res = {}
print('CAMERA (webcam view):')
for name, path in MODELS.items():
    m = Model(path)
    probs, decs = ([], [])
    for p in selected:
        if p['camera'] is None:
            probs.append(None)
            decs.append(None)
            continue
        pr = m.prob(camera_input(p, m.bands))
        probs.append(pr)
        decs.append(1 if pr >= DECISION_THR else 0)
    camera_res[name] = {'prob': probs, 'dec': decs}
    nok = sum((d == p['gt'] for d, p in zip(decs, selected) if d is not None))
    ntot = sum((d is not None for d in decs))
    print(f'  {name:14s} accuracy = {nok / ntot:.3f}  ({ntot} captured)')


6. Results

In [ ]:
import matplotlib.pyplot as plt
names = list(MODELS.keys())
idx = [i for i, p in enumerate(selected) if p['camera'] is not None]

def acc(dec):
    return sum((dec[i] == selected[i]['gt'] for i in idx)) / len(idx)
print(f'scored on {len(idx)}/{len(selected)} captured patches\n')
print(f"{'model':16s}{'direct':>9}{'camera':>9}{'gap':>8}")
for n in names:
    d, c = (acc(direct_res[n]['dec']), acc(camera_res[n]['dec']))
    print(f'{n:16s}{d:9.3f}{c:9.3f}{d - c:+8.3f}')
with open(OUT_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    keys = [n.replace(' ', '_').replace('->', '') for n in names]
    cols = ['stem', 'bucket_pct', 'cloud_frac', 'gt']
    for k in keys:
        cols += [f'{k}_direct_dec', f'{k}_camera_p', f'{k}_camera_dec']
    w.writerow(cols)
    for i, p in enumerate(selected):
        row = [p['stem'], p['bucket'] * 10, f"{p['frac']:.3f}", p['gt']]
        for n in names:
            cp, cd = (camera_res[n]['prob'][i], camera_res[n]['dec'][i])
            row += [direct_res[n]['dec'][i], '' if cp is None else f'{cp:.3f}', '' if cd is None else cd]
        w.writerow(row)
print('saved', OUT_CSV)
x = np.arange(len(names))
bw = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - bw / 2, [acc(direct_res[n]['dec']) for n in names], bw, label='direct')
ax.bar(x + bw / 2, [acc(camera_res[n]['dec']) for n in names], bw, label='camera')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1.05)
ax.set_ylabel('patch accuracy vs ground truth')
ax.legend()
ax.set_title(f'Direct vs camera ({len(idx)} patches)')
for i, n in enumerate(names):
    ax.text(i - bw / 2, acc(direct_res[n]['dec']) + 0.02, f"{acc(direct_res[n]['dec']):.2f}", ha='center')
    ax.text(i + bw / 2, acc(camera_res[n]['dec']) + 0.02, f"{acc(camera_res[n]['dec']):.2f}", ha='center')
plt.tight_layout()
plt.show()


7. Release the camera

In [ ]:
try:
    cap.release()
    print('camera released')
except Exception as e:
    print('already released:', e)
